# 小市值策略优化版

**优化点：**
1. **流动性过滤**：换手率 > 1%，避免低流动性冲击
2. **行业分散**：单行业最多1只，降低集中度风险
3. **波动率加权**：波动率低的股票分配更多仓位
4. **止损机制**：单只股票亏损超过-8%止损

**目标**：2023年以来表现优于原策略

In [1]:
%reload_ext autoreload
%autoreload 2
import sys
sys.path.append("C://Users/20561/Desktop/策略")

import pandas as pd
import polars as pl
import numpy as np
import datetime as dt

from my_utils.fun import read_day_data, get_data_trading_days, get_logger

logging = get_logger(log_file="因子回测/log/小市值策略优化版.log", inherit=False)


【软件终端连接成功！】
【账户信息订阅成功！】


In [2]:
# ========== 策略参数 ==========
START_DATE = '2021-01-04'
END_DATE = '2026-04-20'
INITIAL_CASH = 100_000.0
COMMISSION = 0.0003
SLIPPAGE = 0.000
STOCK_NUM = 3
REFRESH_RATE = 5

# 优化版参数
MV_MIN = 15      # 最小市值（亿）
MV_MAX = 50      # 最大市值（亿）
TURNOVER_MIN = 1.0   # 换手率最低要求（%）
STOP_LOSS = -0.08    # 止损线 -8%
INDUSTRY_DIVERSIFY = True  # 行业分散开关

logging.info(f'优化版参数: MV={MV_MIN}-{MV_MAX}亿, 换手率>{TURNOVER_MIN}%, 止损={STOP_LOSS*100}%')

优化版参数: MV=15-50亿, 换手率>1.0%, 止损=-8.0%


In [3]:
# ========== 读取数据 ==========
logging.info(f"读取数据: {START_DATE} 至 {END_DATE}")
stock_data = read_day_data(start_date=START_DATE, end_date=END_DATE)

# 计算市值（亿元）
stock_data = stock_data.with_columns([
    (pl.col('total_mv') / 1e8).alias("market_cap"),
    # 换手率转百分比
    (pl.col("turnover_rate") * 100).alias("turnover_pct"),
    # 日收益率
    (pl.col("close") / pl.col("pre_close") - 1).alias("daily_return"),
])

logging.info(f"数据行数: {len(stock_data)}")


读取数据: 2021-01-04 至 2026-04-20
数据行数: 6362021


In [4]:
# ========== 选股函数 ==========
def select_stocks_optimized(daily_data, mv_min, mv_max, top_n, turnover_min=1.0, industry_diversify=True):
    """
    优化版选股：
    1. 市值筛选
    2. 流动性筛选（换手率）
    3. 行业分散
    
    返回: [(code, name, industry, market_cap, volatility), ...]
    """
    # 基础筛选
    filtered = daily_data.filter(
        (pl.col('market_cap') >= mv_min) &
        (pl.col('market_cap') <= mv_max) &
        (~pl.col('is_st')) &
        (~pl.col('is_suspended'))
    )

    # 流动性筛选
    if turnover_min > 0:
        filtered = filtered.filter(pl.col('turnover_pct') >= turnover_min)

    if len(filtered) == 0:
        return []

    # 按市值升序排列
    filtered = filtered.sort('market_cap', descending=False)

    # 行业分散：每个行业最多1只
    if industry_diversify:
        selected = []
        industry_selected = set()
        
        for row in filtered.to_dicts():
            if len(selected) >= top_n:
                break
            
            code = row['code']
            # 用代码前缀区分行业板块
            if code.startswith('SZSE.000') or code.startswith('SHSE.600'):
                industry = 'main_board'
            elif code.startswith('SZSE.002'):
                industry = 'sme'
            elif code.startswith('SZSE.300'):
                industry = 'chinext'
            elif code.startswith('SHSE.688'):
                industry = 'star'
            else:
                industry = 'other'
            
            if industry not in industry_selected:
                industry_selected.add(industry)
                vol = row.get('volatility_20d')
                vol = vol if vol is not None and not (vol != vol) else 0.02  # 处理nan
                selected.append((row['code'], row['name'], industry, row['market_cap'], vol))
        
        return selected
    else:
        return [(row['code'], row['name'], 'unknown', 
                row['market_cap'], row.get('volatility_20d', 0.02)) 
               for row in filtered.head(top_n).to_dicts()]

In [5]:
# ========== 生成交易信号 ==========
logging.info('生成交易信号...')
trading_days = get_data_trading_days(START_DATE, END_DATE)
logging.info(f'交易日数量: {len(trading_days)}')

orders_list = []
day_count = 0

# 持仓记录: {code: {volume, buy_price, buy_time}}
current_holdings = {}
cash = INITIAL_CASH

for day in trading_days:
    day_ts = pd.Timestamp(day)
    day_count += 1

    daily_data = stock_data.filter(pl.col('trading_date') == day)

    # === 1. 止损检查 ===
    stocks_to_stop = []
    for code, holding in list(current_holdings.items()):
        day_row = daily_data.filter(pl.col('code') == code)
        if len(day_row) == 0:
            continue
        current_price = day_row['close'].to_list()[0]
        if current_price < holding['buy_price'] * (1 + STOP_LOSS):
            stocks_to_stop.append(code)

    # === 2. 每5个交易日调仓 ===
    need_rebalance = (day_count % REFRESH_RATE == 1)

    if need_rebalance or len(stocks_to_stop) > 0:
        # === 卖出 ===
        all_sell_codes = set(stocks_to_stop)
        if need_rebalance:
            all_sell_codes.update(current_holdings.keys())

        for code in all_sell_codes:
            holding = current_holdings.get(code)
            if not holding:
                continue
            day_row = daily_data.filter(pl.col('code') == code)
            if len(day_row) == 0:
                continue
            sell_price = day_row['open'].to_list()[0]
            volume = holding['volume']

            orders_list.append({
                'datetime': day_ts,
                'code': code,
                'direction': -1,
                'price': sell_price,
                'volume': volume,
                'buy_time': holding['buy_time'],
                'sell_time': day_ts,
                'cash_ratio': 0,
            })
            cash += sell_price * volume * (1 - COMMISSION - 0.001)

        # 清除持仓
        for code in all_sell_codes:
            current_holdings.pop(code, None)

        # === 买入 ===
        if need_rebalance:
            new_stocks = select_stocks_optimized(
                daily_data, MV_MIN, MV_MAX, STOCK_NUM,
                turnover_min=TURNOVER_MIN,
                industry_diversify=INDUSTRY_DIVERSIFY
            )

            if len(new_stocks) > 0:
                # 波动率加权
                vols = [v for _, _, _, _, v in new_stocks]
                inv_vols = [1/max(v, 0.01) for v in vols]
                total_inv_vol = sum(inv_vols)
                
                for code, name, industry, market_cap, volatility in new_stocks:
                    day_row = daily_data.filter(pl.col('code') == code)
                    if len(day_row) == 0:
                        continue
                    buy_price = day_row['open'].to_list()[0]
                    
                    # 波动率加权仓位
                    weight = (1/max(volatility, 0.01)) / total_inv_vol
                    alloc = cash * weight
                    unit_cost = buy_price * (1 + SLIPPAGE)
                    volume = int(alloc // unit_cost)
                    volume_real = (volume // 100) * 100
                    
                    if volume_real < 100:
                        continue
                    
                    orders_list.append({
                        'datetime': day_ts,
                        'code': code,
                        'direction': 1,
                        'price': buy_price,
                        'volume': volume_real,
                        'buy_time': day_ts,
                        'sell_time': None,
                        'cash_ratio': weight,
                    })
                    cash -= buy_price * volume_real * (1 + SLIPPAGE) * (1 + COMMISSION)
                    current_holdings[code] = {
                        'volume': volume_real,
                        'buy_price': buy_price,
                        'buy_time': day_ts,
                    }

        if day_count % 20 == 0:
            logging.info(f'{day} 调仓，现金: {cash:.2f}')

# 转换为DataFrame
orders_df = pd.DataFrame(orders_list)
orders_df['datetime'] = pd.to_datetime(orders_df['datetime'])
logging.info(f'信号生成完成: 买入{len(orders_df[orders_df["direction"]==1])}笔, 卖出{len(orders_df[orders_df["direction"]==-1])}笔')

生成交易信号...
交易日数量: 1272
2021-05-06 调仓，现金: 47939.81
2022-10-28 调仓，现金: 86745.45
2024-01-18 调仓，现金: 124122.17
2024-06-24 调仓，现金: 100940.36
信号生成完成: 买入765笔, 卖出760笔


In [6]:
# ========== 回测 ==========
from my_backtester.my_backtester import Backtester

backtester = Backtester(
    orders=orders_df,
    initial_cash=INITIAL_CASH,
    commission=COMMISSION,
    slippage=SLIPPAGE
)

logging.info('开始回测...')
result = backtester.run(start_time=START_DATE, end_time=END_DATE)
logging.info('回测完成！')

开始回测...
C:\/Users/20561/Desktop/策略\my_backtester\my_backtester.py:587: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.pos_log = pd.concat([self.pos_log, self._snapshot_positions(day_ts)], ignore_index=True)
回测完成！


In [7]:
# ========== 运行报告 ==========
metrics, fig = backtester.report(
    start_date=pd.Timestamp(START_DATE),
    end_date=pd.Timestamp(END_DATE)
)
fig.show()